# Neeko TTS Demo — VoxCPM2 (Türkçe)

**Model:** OpenBMB VoxCPM2 (Apache 2.0 lisans, 2B parametre, 30 dilde resmi Türkçe destek, MiniCPM-4 backbone)

**Donanım:** Kaggle T4 x2 GPU (Settings → Accelerator → **GPU T4 x2** seçili olmalı). VoxCPM2 ~8 GB VRAM ister, T4 (16 GB) rahat sığar.

**Çıktı:** Chatterbox demosuyla **aynı 10 Türkçe çocuk cümlesi**, `neeko-demo-v0.1-voxcpm2.zip` paketi. Yan yana dinleme için iki dosya seti karşılaştırılacak.

**Tahmini süre:** ~12-18 dakika (model 2B parametre, ilk indirme ~5-8 dk + 10 cümle ~4-6 dk).

**Repo:** [`neeko-voice/notebooks/01-voxcpm2-tr-demo.ipynb`](https://github.com/)

**Not:** Bu notebook `00-chatterbox-tr-demo.ipynb` ile aynı cümle setini, aynı çıktı şemasını kullanır. Tek fark: model katmanı. Damıtmadaki aday yarışının ikinci ayağı.

## 1. Kurulum
VoxCPM paketini kur. `voxcpm` paketi hem VoxCPM (eski) hem VoxCPM2 (yeni) modellerini destekler.

In [ ]:
!pip install -q voxcpm soundfile

## 2. Donanım doğrulama

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM (toplam): {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("UYARI: GPU yok. Settings -> Accelerator -> GPU T4 x2 seç ve session'ı restart et.")

## 3. Model yükle
VoxCPM2'yi Hugging Face'ten (`openbmb/VoxCPM2`) indir. 2B parametre, ~5-8 GB indirme. İlk koşuda ~5-8 dakika sürer.

`load_denoiser=False` — opsiyonel post-processing denoiser kapalı (ekstra ağırlık + süre).

In [ ]:
from voxcpm import VoxCPM
import time

t0 = time.time()
model = VoxCPM.from_pretrained(
    "openbmb/VoxCPM2",
    load_denoiser=False,
)
print(f"Model yüklendi: {time.time() - t0:.1f} saniye")
sample_rate = model.tts_model.sample_rate
print(f"Sample rate: {sample_rate} Hz")

## 4. Test cümleleri
Chatterbox demosuyla **bire bir aynı** set — yan yana karşılaştırma için sabit kalmalı (repo: `data/test-sets/v0.1-mini.md`).

In [ ]:
test_sentences = [
    ("01", "oyun",        "Hadi birlikte sıradaki hayvanı bulalım!"),
    ("02", "uyku",        "Şimdi gözlerini kapatıp derin bir nefes alalım."),
    ("03", "ders",        "Yedi artı üç kaç eder, biliyor musun?"),
    ("04", "sefkat",      "Kızgın olman çok normal, önce sakinleşelim."),
    ("05", "soru",        "Sence bugün ne renk bir gökyüzü gördük?"),
    ("06", "heyecan",     "Vay canına, çok güzel bir resim çizmişsin!"),
    ("07", "sayi_tarih",  "Yirmi üç Nisan'da ne kutlarız?"),
    ("08", "kisaltma",    "Doktor Ayşe, bunu bir büyüğünle birlikte yapalım dedi."),
    ("09", "kod_karisim", "Annenin iPhone'unu yerine bırakır mısın?"),
    ("10", "hikaye",      "Bir varmış, bir yokmuş, çok uzak bir ülkede küçük bir tavşan yaşarmış."),
]

for idx, cat, text in test_sentences:
    print(f"{idx} [{cat:12s}] {text}")

## 5. Sentezle ve kaydet
VoxCPM2'nin `generate` API'si: `text` + `cfg_value` (classifier-free guidance, 1.0-3.0 tipik; default 2.0) + `inference_timesteps` (kalite/hız trade-off, 10 makul).

Model multilingual — Türkçe metin doğrudan verilir, dil otomatik tespit edilir.

In [ ]:
import os
import soundfile as sf

out_dir = "/kaggle/working/output"
os.makedirs(out_dir, exist_ok=True)

results = []
for idx, cat, text in test_sentences:
    t0 = time.time()
    wav = model.generate(
        text=text,
        cfg_value=2.0,
        inference_timesteps=10,
    )
    elapsed = time.time() - t0
    
    out_path = f"{out_dir}/{idx}_{cat}.wav"
    sf.write(out_path, wav, sample_rate)
    
    duration = len(wav) / sample_rate
    rtf = elapsed / duration if duration > 0 else float('inf')
    results.append({"id": idx, "cat": cat, "text": text, "duration_s": duration, "elapsed_s": elapsed, "rtf": rtf})
    print(f"{idx} [{cat:12s}] {duration:.2f}s ses, {elapsed:.2f}s işlem, RTF={rtf:.2f}")

print(f"\nToplam: {len(results)} dosya kaydedildi -> {out_dir}")

## 6. Metadata yaz

In [ ]:
import json
from datetime import datetime

metadata = {
    "model": "openbmb/VoxCPM2",
    "model_params": "2B",
    "language": "tr",
    "sample_rate_hz": int(sample_rate),
    "device": device,
    "cfg_value": 2.0,
    "inference_timesteps": 10,
    "date_utc": datetime.utcnow().isoformat(),
    "test_set_version": "v0.1-mini",
    "sentences": results,
    "mean_rtf": sum(r["rtf"] for r in results) / len(results),
}

with open(f"{out_dir}/metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(json.dumps(metadata, ensure_ascii=False, indent=2))

## 7. Paketle

In [ ]:
import shutil
zip_path = "/kaggle/working/neeko-demo-v0.1-voxcpm2"
shutil.make_archive(zip_path, "zip", out_dir)
print(f"Paket hazır: {zip_path}.zip")
print("\nİndirme: Sağ panel -> Output -> neeko-demo-v0.1-voxcpm2.zip")

## 8. Hızlı dinleme (opsiyonel)

In [ ]:
from IPython.display import Audio, display, Markdown

for idx, cat, text in test_sentences:
    display(Markdown(f"**{idx} [{cat}]** — {text}"))
    display(Audio(f"{out_dir}/{idx}_{cat}.wav"))

## 9. cfg_value tuning (opsiyonel deney)
`cfg_value` parametresi kalite/çeşitlilik dengesi. Yüksek değerler (2.5-3.0) daha 'sıkı' okur ama monotonlaşabilir; düşük değerler (1.0-1.5) daha doğal ama tutarsız olabilir. Bir cümle üzerinde 3 farklı değer dene.

In [ ]:
tuning_text = test_sentences[5][2]  # 'Vay canına, çok güzel bir resim çizmişsin!'
tuning_dir = f"{out_dir}/cfg_tuning"
os.makedirs(tuning_dir, exist_ok=True)

for cfg in [1.5, 2.0, 2.5, 3.0]:
    wav = model.generate(text=tuning_text, cfg_value=cfg, inference_timesteps=10)
    out_path = f"{tuning_dir}/heyecan_cfg{cfg}.wav"
    sf.write(out_path, wav, sample_rate)
    display(Markdown(f"**cfg_value={cfg}**"))
    display(Audio(out_path))

---

## Sonraki adımlar

1. Bu .zip'i indir, `experiments/2026-05-19-voxcpm2-baseline/output/` altına aç
2. Chatterbox çıktısıyla yan yana koy — aynı cümleyi iki modelde dinle
3. Her cümle için iki modelin puanlarını `experiments/2026-05-19-voxcpm2-baseline/log.md` altında tabloya yaz
4. Atlas damıtmaya bu çapraz-karşılaştırmayı dahil eder → "birincil model adayı" satırı veriyle kapanır